[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/practicas/Practica_4.ipynb)

# Práctica 4: Implementación del CAPM

## Objetivo de aprendizaje
Aplicar una regresión lineal simple e interpretar correctamente
- La R cuadrada como medida de ajuste
- La significancia global del modelo
- La significancia individual del coeficiente
- La interpretación económica del coeficiente beta

## Estructura del ejercicio
En la sección de "Contexto y descarga de datos" únicamente debes
- Sustituir el ticker de la acción por uno diferente al del ejemplo
- Sustituir el ticker del mercado si es necesario.

En la sección de "Análisis e interpretación:
- Es donde debes escribir tu código adicional
- Es donde debes realizar la interpretación económica y estadística del modelo.

## Contexto y descarga de datos
En esta práctica utilizaremos un modelo de regresión para implementar una versión simplificada del CAPM (Capital Asset Pricing Model) y estimar la *beta financiera* de una acción. Este modelo implica que los rendimientos esperados de un acción está determinado por los rendimientos del mercado y la tasa libre de riesgo.  

$E(R_i) = R_f + \beta_i (E(R_m) - R_f)$

Si reordenamos esta expresión y agregamos un intercepto obtendremos una ecuación que se puede estimar con una regresión lineal:

$R_i - R_f = \alpha + \beta_i (R_m - R_f) + \epsilon$

En este ejercicio, por practicidad, vamos a correr este modelo de manera directa omitiendo la tasa libre de riesgo:

$R_i = \alpha + \beta_i R_m + \epsilon$

> **Nota: alcance didáctico del ejercicio.** La versión del CAPM que estimarás aquí es una simplificación con fines de aprendizaje. Antes de interpretar los resultados, ten presente que:
> - Se omite la tasa libre de riesgo, por lo que el intercepto no debe leerse como el *alfa de Jensen* en sentido estricto.
> - Se usa un índice bursátil como aproximación del "mercado", cuando en teoría el CAPM se refiere al portafolio de mercado completo.
> - La beta estimada depende del periodo, la frecuencia de los datos y el índice elegidos: no es un parámetro único ni definitivo de la acción.
> - El objetivo es practicar la estimación e interpretación de una regresión, no generar un insumo para una decisión real de inversión.
>
> Estas limitaciones no invalidan el ejercicio, pero sí acotan las conclusiones que puedes defender.

In [ ]:
# Importar bibliotecas
import pandas as pd
import yfinance as yf
import numpy as np
import statsmodels.api as sm

Definimos el periodo de análisis, la acción que vamos a estudiar, el índice de mercado que usaremos como referencia. Si tu acción es de una empresa industrial usa '^DJI' como mercado, si es tecnológica, utiliza '^IXIC'
**Importante: Utiliza los datos de una acción diferente a la del ejemplo ('AAPL')**

In [ ]:
# Parámetros
inicio = '2023-01-01'
fin = '2026-02-04'
ticker = 'AAPL'    # Cambiar por otra acción
market = '^DJI'    # Cambiar si se requiere

# Descargar datos
datos = yf.download([ticker, market], start=inicio, end=fin, progress=False)

Luego, seleccionamos los precios relevantes, renombramos las columnas por comodidad y dejamos una estructura clara

In [ ]:
# Seleccionar precios ajustados
if 'Adj Close' in datos.columns:
    precios = datos['Adj Close']
else:
    precios = datos['Close']  # respaldo si Adj Close no está disponible

# Renombrar columnas
precios = precios.rename(columns={ticker: 'accion', market: 'mercado'})

precios.head()

El CAPM se estima normalmente con rendimientos, no con precios. Por ello, 
1. Convertimos los datos a frecuencia mensual
2. Calculamos el rendimiento como porcentaje de cambio
3. Eliminamos observaciones con valores faltantes.

In [ ]:
# Convertir a datos mensuales (último día del mes)
precios_mensuales = precios.resample('ME').last()

# Calcular rendimientos
rendimientos = precios_mensuales.pct_change()

# Eliminar NA
rendimientos = rendimientos.dropna()

rendimientos.head()

## Análisis e interpretación
Calcula un modelo de regresión con el rendimiento de la acción como variable dependiente y el rendimiento del mercado como variable independiente (no olvides incluir el intercepto). Una vez estimada la regresión, responde con explicación en lenguaje claro (no solo copiar valores):
- *El ajuste del modelo (R cuadrada).* ¿qué porcentaje de la variación del rendimiento de la acción es explicado por el modelo? ¿consideras que el ajuste es alto, moderado o bajo? ¿qué implicaciones tiene esto desde el punto de vista financiero?
- *El p-valor del estadístico F.* ¿el modelo en conjunto es estadísticamente significativo? ¿qué hipótesis está evaluando el estadístico F? Interpreta el p-valor en términos prácticos considerando un nivel de significancia del 5%.
- *El p-valor de la variable independiente.* ¿La beta es estadísticamente diferente de cero? ¿qué implicaría que no fuera significativa?
- *El coeficiente de regresión de la variable independiente (beta financiera)*. Interpreta el valor estimado. ¿La acción es más o menos volátil que el mercado? ¿Qué tipo de inversionista podría sentirse atraído por esta acción?

## Uso de IA generativa (para extender, no para resolver)

La IA se usa para **ampliar** la práctica partiendo de lo que ya construiste, no para hacerla. Declara la herramienta y la versión utilizada (por ejemplo, ChatGPT 5, Claude Opus 4.5, Gemini 3 Pro, Copilot).

**Qué debes entregar** (cuatro bloques, en celdas de texto dentro del notebook):

1. **Tu pregunta de negocio.** Una pregunta propia, pertinente y que la práctica **no** responda. Formúlala en primera persona: "Quiero saber si...", "Me preocupa que...". No se acepta reproducir la pregunta del ejercicio ni preguntas genéricas. Del tipo esperado (no para copiar): *"Quiero saber si la beta de mi acción cambió a partir de un evento concreto de la empresa, porque eso afectaría cómo la clasifico por riesgo"*.
2. **El prompt completo, transcrito en celda de texto** (no de código). Debe incluir: rol, objetivo, tu código, **los resultados reales de tu regresión pegados** (la salida de `summary()`) y una restricción explícita de lo obvio (por ejemplo: "no me expliques qué es la beta ni qué es el CAPM"). Un prompt de una línea, sin contexto ni resultados, no cuenta.
3. **Qué adopté y qué descarté.** De la respuesta recibida, indica qué implementaste y qué dejaste fuera, con la razón. La implementación usa **máximo 50 líneas de código**.
4. **Verificación.** Un cálculo, contraejemplo o comprobación que valide o refute algo que la IA afirmó. Por ejemplo: si la IA afirma que la beta es estable en el periodo, divide la muestra en dos subperiodos y reestima; si sugiere que el índice de mercado elegido altera el resultado, reestima con otro índice y compara.

**Evidencia auditable**: pega el resultado completo de la extensión (texto, tablas o código), no un enlace a la conversación. Revisa que el documento o el código no quede cortado.

**No se acepta**:
- Transferir las instrucciones de la práctica a la IA, ya sea copiándolas o parafraseándolas, para que ella la resuelva.
- Usar la IA como enciclopedia: respuestas generales sin tus datos ni tu código. Está bien usarla para comprender, pero debe haber contenido propio nuevo.
- Transcribir sugerencias sin implementarlas, o dar por cierto lo que la IA afirma sin comprobarlo.
- Delegar la interpretación: las conclusiones y su redacción son tuyas.

**Buenas prácticas sugeridas**: repreguntar a la IA sobre su propia respuesta; pedirle explícitamente las limitaciones de lo que propone; traer un concepto externo al curso y aplicarlo a tus datos.

**Defensa oral**: cualquier práctica puede ser seleccionada al azar para una defensa oral breve (3 a 5 minutos), en la que deberás explicar tus decisiones, tu código y tus conclusiones. Un trabajo que no pueda ser explicado por su autor se considerará evidencia de trabajo no auténtico y podrá ser penalizado.

## Entregable 
Notebook en Jupyter exportado a pdf o html, con el código, análisis e interpretación. 

## Rúbrica de evaluación
1. Implementación técnica (20%): descarga y limpieza correcta de datos, cálculo correcto de rendimientos, modelo estimado correctamente.
2. Interpretación de R cuadrada (10%): interpreta correctamente y discute implicación económica. Se penalizará que sólo reporte el número sin interpretación real.
3. Significancia del modelo (10%): Interpreta p-valor y concluye correctamente. Se penalizará que sólo indique si es significativo o no.
4. Significancia de la beta (10%): Interpreta p-valor y conecta con significado financiero. Se penalizará que sólo indique si es significativo o no.
5. Interpretación económica de la beta (15%): Explica el riesgo sistemático y compara con el mercado. Se penalizará que sólo describa el número.
6. Claridad y redacción (15%): Explicación clara, lenguaje técnico correcto, coherencia. Se penalizará respuestas vagas o confusas.
7. **Extensión del análisis con IA generativa (20%)**, evaluada en cuatro partes iguales (5% cada una):
   - *Pregunta propia (5%)*: pregunta de negocio formulada por el alumno, pertinente y no respondida por la práctica. Se anula si reproduce la pregunta del ejercicio o si es genérica.
   - *Calidad del prompt (5%)*: transcrito completo en celda de texto, con rol, objetivo, código y los resultados reales de la regresión, más una restricción explícita de lo obvio. Se anula si se transfieren las instrucciones de la práctica (copiadas o parafraseadas).
   - *Ejecución: qué adopté y qué descarté (5%)*: lo propuesto se implementa (máximo 50 líneas de código) y se explica qué quedó fuera y por qué. No basta transcribir sugerencias.
   - *Verificación e interpretación propia (5%)*: un cálculo, contraejemplo o comprobación que valide o refute una afirmación de la IA, y conclusiones redactadas por el alumno.

   *Requisito de forma*: la evidencia debe ser auditable, esto es, resultado completo pegado en el notebook, sin cortes y sin enlaces a la conversación como único respaldo.